In [14]:
import pandas as pd
import numpy as np
import tkinter as tk
from tkinter import ttk, messagebox

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [15]:
df = pd.read_csv(r"C:\Users\Admin\Desktop\dataset\smart_traffic_congestion_dataset.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully!
Dataset shape: (1200, 12)


,Date,Time,Day,Temperature,Humidity,Rainfall,Holiday,Vehicle_Count,Average_Speed,Road_Occupancy,Accident,Congestion_Level
0,2026-03-02,04:00,Monday,23.3,68.6,4.5,No,422,48.4,28.4,No,Low
1,2026-01-07,20:45,Wednesday,28.6,68.3,0.0,No,532,45.9,36.8,No,Medium
2,2026-03-12,16:45,Thursday,26.3,57.7,0.0,No,319,53.7,25.1,Yes,Medium
3,2026-03-02,10:15,Monday,28.7,60.7,0.0,No,815,25.8,61.4,No,Medium
4,2026-02-13,09:15,Friday,26.8,69.7,0.0,No,1448,15.0,100.0,No,High


In [16]:
# Convert Date
df["Date"] = pd.to_datetime(df["Date"])

# Extract date features
df["Day_of_Week"] = df["Date"].dt.dayofweek
df["Month"] = df["Date"].dt.month

# Convert Time
time_data = pd.to_datetime(
    df["Time"],
    format="%H:%M"
)

df["Hour"] = time_data.dt.hour

# Peak hour
df["Peak_Hour"] = df["Hour"].apply(
    lambda x: 1 if x in [7, 8, 9, 17, 18, 19] else 0
)

df.head()

,Date,Time,Day,Temperature,Humidity,Rainfall,Holiday,Vehicle_Count,Average_Speed,Road_Occupancy,Accident,Congestion_Level,Day_of_Week,Month,Hour,Peak_Hour
0,2026-03-02,04:00,Monday,23.3,68.6,4.5,No,422,48.4,28.4,No,Low,0,3,4,0
1,2026-01-07,20:45,Wednesday,28.6,68.3,0.0,No,532,45.9,36.8,No,Medium,2,1,20,0
2,2026-03-12,16:45,Thursday,26.3,57.7,0.0,No,319,53.7,25.1,Yes,Medium,3,3,16,0
3,2026-03-02,10:15,Monday,28.7,60.7,0.0,No,815,25.8,61.4,No,Medium,0,3,10,0
4,2026-02-13,09:15,Friday,26.8,69.7,0.0,No,1448,15.0,100.0,No,High,4,2,9,1


In [17]:
# Holiday encoding
holiday_encoder = LabelEncoder()

df["Holiday"] = holiday_encoder.fit_transform(
    df["Holiday"]
)

# Accident encoding
accident_encoder = LabelEncoder()

df["Accident"] = accident_encoder.fit_transform(
    df["Accident"]
)

# Target encoding
target_encoder = LabelEncoder()

df["Congestion_Level"] = target_encoder.fit_transform(
    df["Congestion_Level"]
)

print("Traffic classes:")

for i, name in enumerate(target_encoder.classes_):
    print(i, "=", name)

Traffic classes:
0 = High
1 = Low
2 = Medium


In [18]:
features = [
    "Temperature",
    "Humidity",
    "Rainfall",
    "Holiday",
    "Vehicle_Count",
    "Average_Speed",
    "Road_Occupancy",
    "Accident",
    "Day_of_Week",
    "Month",
    "Hour",
    "Peak_Hour"
]

X = df[features]

y = df["Congestion_Level"]

print("Number of features:", len(features))
print(features)

Number of features: 12
['Temperature', 'Humidity', 'Rainfall', 'Holiday', 'Vehicle_Count', 'Average_Speed', 'Road_Occupancy', 'Accident', 'Day_of_Week', 'Month', 'Hour', 'Peak_Hour']


In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (960, 12)
Testing data: (240, 12)


In [20]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

print("Random Forest model trained successfully!")

Random Forest model trained successfully!


In [21]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    y_pred
)

print(
    "Model Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

Model Accuracy: 81.67 %


In [22]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=target_encoder.classes_
    )
)

              precision    recall  f1-score   support

        High       0.93      0.90      0.92        60
         Low       0.82      0.82      0.82        96
      Medium       0.73      0.75      0.74        84

    accuracy                           0.82       240
   macro avg       0.83      0.82      0.83       240
weighted avg       0.82      0.82      0.82       240



In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
import pandas as pd


# =========================================================
# PREDICTION FUNCTION
# =========================================================

def predict_traffic():

    try:
        # Get input values
        date_value = date_entry.get().strip()
        time_value = time_entry.get().strip()

        temperature = float(temperature_entry.get())
        humidity = float(humidity_entry.get())
        rainfall = float(rainfall_entry.get())

        vehicle_count = int(vehicle_entry.get())
        average_speed = float(speed_entry.get())
        road_occupancy = float(occupancy_entry.get())

        holiday = holiday_combo.get()
        accident = accident_combo.get()

        # Check date and time
        date = pd.to_datetime(date_value)

        time = pd.to_datetime(
            time_value,
            format="%H:%M"
        )

        # Extract features
        day_of_week = date.dayofweek
        month = date.month
        hour = time.hour

        peak_hour = 1 if hour in [
            7, 8, 9, 17, 18, 19
        ] else 0

        # Encode Yes/No
        holiday_encoded = 1 if holiday == "Yes" else 0
        accident_encoded = 1 if accident == "Yes" else 0

        # Create dataframe
        new_data = pd.DataFrame({

            "Temperature": [temperature],
            "Humidity": [humidity],
            "Rainfall": [rainfall],
            "Holiday": [holiday_encoded],
            "Vehicle_Count": [vehicle_count],
            "Average_Speed": [average_speed],
            "Road_Occupancy": [road_occupancy],
            "Accident": [accident_encoded],
            "Day_of_Week": [day_of_week],
            "Month": [month],
            "Hour": [hour],
            "Peak_Hour": [peak_hour]

        })

        # Prediction
        prediction = model.predict(new_data)

        prediction_name = target_encoder.inverse_transform(
            prediction
        )[0]

        # Probability
        probabilities = model.predict_proba(new_data)[0]

        probability_dict = dict(
            zip(
                target_encoder.classes_,
                probabilities
            )
        )

        # Display result
        result_label.config(
            text=prediction_name.upper()
        )

        # Display probability
        low_probability = probability_dict.get(
            "Low", 0
        ) * 100

        medium_probability = probability_dict.get(
            "Medium", 0
        ) * 100

        high_probability = probability_dict.get(
            "High", 0
        ) * 100

        low_label.config(
            text=f"Low       : {low_probability:.2f}%"
        )

        medium_label.config(
            text=f"Medium : {medium_probability:.2f}%"
        )

        high_label.config(
            text=f"High      : {high_probability:.2f}%"
        )

        # Result color
        if prediction_name == "High":

            result_label.config(
                fg="#ef4444"
            )

        elif prediction_name == "Medium":

            result_label.config(
                fg="#f59e0b"
            )

        else:

            result_label.config(
                fg="#22c55e"
            )

    except Exception as e:

        messagebox.showerror(
            "Prediction Error",
            str(e)
        )


# =========================================================
# RESET
# =========================================================

def reset_values():

    date_entry.delete(0, tk.END)
    date_entry.insert(0, "2026-09-23")

    time_entry.delete(0, tk.END)
    time_entry.insert(0, "08:30")

    temperature_entry.delete(0, tk.END)
    temperature_entry.insert(0, "28")

    humidity_entry.delete(0, tk.END)
    humidity_entry.insert(0, "65")

    rainfall_entry.delete(0, tk.END)
    rainfall_entry.insert(0, "0")

    vehicle_entry.delete(0, tk.END)
    vehicle_entry.insert(0, "1200")

    speed_entry.delete(0, tk.END)
    speed_entry.insert(0, "25")

    occupancy_entry.delete(0, tk.END)
    occupancy_entry.insert(0, "80")

    holiday_combo.set("No")
    accident_combo.set("No")

    result_label.config(
        text="---",
        fg="white"
    )

    low_label.config(
        text="Low       : 0.00%"
    )

    medium_label.config(
        text="Medium : 0.00%"
    )

    high_label.config(
        text="High      : 0.00%"
    )


# =========================================================
# WINDOW
# =========================================================

root = tk.Tk()

root.title(
    "Smart Traffic Congestion Prediction"
)

root.geometry(
    "950x650"
)

root.configure(
    bg="#0f172a"
)


# =========================================================
# TITLE
# =========================================================

tk.Label(
    root,
    text="🚦 SMART TRAFFIC CONGESTION",
    font=("Arial", 25, "bold"),
    bg="#0f172a",
    fg="white"
).pack(pady=(20, 5))


tk.Label(
    root,
    text="Machine Learning Traffic Prediction System",
    font=("Arial", 12),
    bg="#0f172a",
    fg="#94a3b8"
).pack(
    pady=(0, 20)
)


# =========================================================
# MAIN FRAME
# =========================================================

main_frame = tk.Frame(
    root,
    bg="#1e293b"
)

main_frame.pack(
    padx=30,
    pady=10,
    fill="both",
    expand=True
)


# =========================================================
# LEFT SIDE
# =========================================================

left_frame = tk.Frame(
    main_frame,
    bg="#1e293b"
)

left_frame.pack(
    side="left",
    fill="both",
    expand=True,
    padx=30,
    pady=20
)


tk.Label(
    left_frame,
    text="TRAFFIC INFORMATION",
    font=("Arial", 14, "bold"),
    bg="#1e293b",
    fg="#38bdf8"
).pack(
    anchor="w",
    pady=(0, 15)
)


# =========================================================
# HELPER
# =========================================================

def add_input(label_text, default_value):

    frame = tk.Frame(
        left_frame,
        bg="#1e293b"
    )

    frame.pack(
        fill="x",
        pady=5
    )

    tk.Label(
        frame,
        text=label_text,
        width=22,
        anchor="w",
        font=("Arial", 10),
        bg="#1e293b",
        fg="white"
    ).pack(
        side="left"
    )

    entry = tk.Entry(
        frame,
        font=("Arial", 10),
        width=22
    )

    entry.pack(
        side="right",
        ipady=5
    )

    entry.insert(
        0,
        default_value
    )

    return entry


# =========================================================
# INPUTS
# =========================================================

date_entry = add_input(
    "Date (YYYY-MM-DD)",
    "2026-09-23"
)

time_entry = add_input(
    "Time (HH:MM)",
    "08:30"
)

temperature_entry = add_input(
    "Temperature (°C)",
    "28"
)

humidity_entry = add_input(
    "Humidity (%)",
    "65"
)

rainfall_entry = add_input(
    "Rainfall (mm)",
    "0"
)

vehicle_entry = add_input(
    "Vehicle Count",
    "1200"
)

speed_entry = add_input(
    "Average Speed (km/h)",
    "25"
)

occupancy_entry = add_input(
    "Road Occupancy (%)",
    "80"
)


# =========================================================
# HOLIDAY
# =========================================================

holiday_frame = tk.Frame(
    left_frame,
    bg="#1e293b"
)

holiday_frame.pack(
    fill="x",
    pady=5
)

tk.Label(
    holiday_frame,
    text="Holiday",
    width=22,
    anchor="w",
    font=("Arial", 10),
    bg="#1e293b",
    fg="white"
).pack(
    side="left"
)

holiday_combo = ttk.Combobox(
    holiday_frame,
    values=["No", "Yes"],
    state="readonly",
    width=20
)

holiday_combo.pack(
    side="right"
)

holiday_combo.set("No")


# =========================================================
# ACCIDENT
# =========================================================

accident_frame = tk.Frame(
    left_frame,
    bg="#1e293b"
)

accident_frame.pack(
    fill="x",
    pady=5
)

tk.Label(
    accident_frame,
    text="Accident",
    width=22,
    anchor="w",
    font=("Arial", 10),
    bg="#1e293b",
    fg="white"
).pack(
    side="left"
)

accident_combo = ttk.Combobox(
    accident_frame,
    values=["No", "Yes"],
    state="readonly",
    width=20
)

accident_combo.pack(
    side="right"
)

accident_combo.set("No")


# =========================================================
# BUTTONS
# =========================================================

button_frame = tk.Frame(
    left_frame,
    bg="#1e293b"
)

button_frame.pack(
    pady=20
)


predict_button = tk.Button(
    button_frame,
    text="🚦  PREDICT TRAFFIC",
    command=predict_traffic,
    font=("Arial", 12, "bold"),
    bg="#38bdf8",
    fg="#0f172a",
    activebackground="#7dd3fc",
    activeforeground="#0f172a",
    relief="flat",
    cursor="hand2",
    padx=20,
    pady=10
)

predict_button.pack(
    side="left",
    padx=5
)


reset_button = tk.Button(
    button_frame,
    text="↻  RESET",
    command=reset_values,
    font=("Arial", 11, "bold"),
    bg="#334155",
    fg="white",
    activebackground="#475569",
    relief="flat",
    cursor="hand2",
    padx=20,
    pady=10
)

reset_button.pack(
    side="left",
    padx=5
)


# =========================================================
# RIGHT SIDE
# =========================================================

right_frame = tk.Frame(
    main_frame,
    bg="#263449"
)

right_frame.pack(
    side="right",
    fill="both",
    expand=True,
    padx=30,
    pady=30
)


tk.Label(
    right_frame,
    text="PREDICTION RESULT",
    font=("Arial", 14, "bold"),
    bg="#263449",
    fg="#38bdf8"
).pack(
    pady=(25, 15)
)


result_label = tk.Label(
    right_frame,
    text="---",
    font=("Arial", 38, "bold"),
    bg="#263449",
    fg="white"
)

result_label.pack(
    pady=30
)


tk.Label(
    right_frame,
    text="CONGESTION PROBABILITY",
    font=("Arial", 11, "bold"),
    bg="#263449",
    fg="white"
).pack(
    pady=10
)


low_label = tk.Label(
    right_frame,
    text="Low       : 0.00%",
    font=("Arial", 12),
    bg="#263449",
    fg="#22c55e"
)

low_label.pack(
    pady=10
)


medium_label = tk.Label(
    right_frame,
    text="Medium : 0.00%",
    font=("Arial", 12),
    bg="#263449",
    fg="#f59e0b"
)

medium_label.pack(
    pady=10
)


high_label = tk.Label(
    right_frame,
    text="High      : 0.00%",
    font=("Arial", 12),
    bg="#263449",
    fg="#ef4444"
)

high_label.pack(
    pady=10
)


# =========================================================
# MODEL INFO
# =========================================================

tk.Label(
    right_frame,
    text="MODEL",
    font=("Arial", 9, "bold"),
    bg="#263449",
    fg="#94a3b8"
).pack(
    pady=(30, 5)
)


tk.Label(
    right_frame,
    text="Random Forest Classifier",
    font=("Arial", 11, "bold"),
    bg="#263449",
    fg="white"
).pack()


tk.Label(
    right_frame,
    text=f"Accuracy: {accuracy * 100:.2f}%",
    font=("Arial", 10),
    bg="#263449",
    fg="#38bdf8"
).pack(
    pady=5
)


# =========================================================
# RUN
# =========================================================

root.mainloop()

In [ ]:
print(
    "Model Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

In [ ]:
print(model)

In [ ]:
import tkinter as tk

test = tk.Tk()
test.title("Button Test")
test.geometry("300x200")

def test_button():
    label.config(text="BUTTON WORKS!")

tk.Button(
    test,
    text="CLICK ME",
    command=test_button,
    font=("Arial", 14)
).pack(pady=40)

label = tk.Label(
    test,
    text="Waiting..."
)

label.pack()

test.mainloop()